In [ ]:
from pathlib import Path
import sys

RAIZ = Path.cwd().resolve()
if not (RAIZ / "src" / "carregar_dados.py").is_file():
    RAIZ = RAIZ.parent
if not (RAIZ / "src" / "carregar_dados.py").is_file():
    raise RuntimeError("Abra o notebook na raiz do repositório ou em notebooks/.")
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))


In [ ]:
import pandas as pd
from src.carregar_dados import carregar_dados

df = carregar_dados()
df.head()


In [ ]:
categoricas = [
    "bairro",
    "padrao_acabamento",
    "tipo_construcao",
    "tipo_ocupacao",
    "estado_conservacao",
    "tipo_imovel",
    "cidade",
    "uf"
]

cardinalidade = (
    df[categoricas]
    .nunique(dropna=True)
    .to_frame(name="cardinalidade")
    .sort_values("cardinalidade", ascending=False)
)

display(cardinalidade)

In [ ]:
for coluna in categoricas:
    frequencia = df[coluna].value_counts(dropna=False)

    tabela = pd.DataFrame({
        "quantidade": frequencia,
        "percentual": (frequencia / len(df) * 100).round(2)
    })

    print(f"\nColuna: {coluna}")
    display(tabela)

In [ ]:
import unicodedata

def normalizar_texto(texto):
    texto = unicodedata.normalize("NFKD", str(texto))
    texto = "".join(
        caractere for caractere in texto
        if not unicodedata.combining(caractere)
    )
    return " ".join(texto.casefold().split())

for coluna in categoricas:
    valores = pd.DataFrame({
        "original": df[coluna].dropna().unique()
    })

    valores["normalizado"] = valores["original"].map(normalizar_texto)

    variantes = (
        valores.groupby("normalizado")["original"]
        .agg(list)
    )

    variantes = variantes[variantes.map(len) > 1]

    print(f"\nPossíveis grafias duplicadas em {coluna}:")
    display(variantes.to_frame("grafias"))

    texto = df[coluna].astype("string")
    espacos = texto.str.contains(r"^\s|\s$|\s{2,}", na=False)

    print("Valores com espaços extras:")
    display(df.loc[espacos, coluna].value_counts())

In [ ]:
eda = df.copy()

for coluna in categoricas:
    eda[coluna] = (
        eda[coluna].astype("string")
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

eda["bairro"] = eda["bairro"].replace({
    "Gracas": "Graças",
    "Santo Antonio": "Santo Antônio",
    "Sao Jose": "São José"
})

display(pd.DataFrame({
    "cardinalidade_original": df[categoricas].nunique(),
    "cardinalidade_padronizada": eda[categoricas].nunique()
}))

In [ ]:
base_faltantes = df.replace(r"^\s*$", pd.NA, regex=True)

faltantes = pd.DataFrame({
    "quantidade faltante": base_faltantes.isna().sum(),
    "percentual faltante": base_faltantes.isna().mean().mul(100)
}).sort_values("percentual faltante", ascending=False)

display(faltantes.round(2))

In [ ]:
categoricas_analise = [
    "bairro",
    "padrao_acabamento",
    "tipo_construcao",
    "tipo_ocupacao",
    "estado_conservacao",
    "tipo_imovel"
]

for coluna in categoricas_analise:
    resumo = (
        eda.groupby(coluna)["valor_avaliacao"]
        .agg(
            quantidade="count",
            mediana="median",
            q10=lambda valores: valores.quantile(0.10),
            q25=lambda valores: valores.quantile(0.25),
            q75=lambda valores: valores.quantile(0.75),
            q90=lambda valores: valores.quantile(0.90)
        )
        .sort_values("quantidade", ascending=False)
    )

    print(f"\nValor de avaliação por {coluna}:")
    display(resumo.round(2))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Identifica nulos e textos vazios.
faltantes = eda.isna()

for coluna in eda.select_dtypes(include=["object", "string"]).columns:
    vazios = eda[coluna].astype("string").str.strip().eq("")
    faltantes[coluna] = faltantes[coluna] | vazios.fillna(False)

In [ ]:
percentuais = (
    faltantes.mean().mul(100)
    .loc[lambda s: s > 0]
    .sort_values()
)

ax = percentuais.plot.barh(figsize=(8, 4), color="steelblue")

for i, valor in enumerate(percentuais):
    ax.text(valor + 0.5, i, f"{valor:.2f}%", va="center")

ax.set_xlim(0, percentuais.max() + 8)
ax.set_title("Percentual de valores faltantes por coluna")
ax.set_xlabel("Registros com valor faltante (%)")
ax.set_ylabel("Coluna")

plt.tight_layout()
plt.show()

In [ ]:
contagens = (
    eda["bairro"].value_counts()
    .head(15)
    .sort_values()
)

ax = contagens.plot.barh(figsize=(9, 6), color="steelblue")

for i, valor in enumerate(contagens):
    ax.text(valor + 20, i, str(valor), va="center")

ax.set_xlim(0, contagens.max() * 1.15)
ax.set_title("15 bairros com mais registros")
ax.set_xlabel("Quantidade de registros")
ax.set_ylabel("Bairro")

plt.tight_layout()
plt.show()

In [ ]:
contagens = eda["tipo_imovel"].value_counts().sort_values()

ax = contagens.plot.barh(figsize=(10, 8), color="steelblue")

for i, valor in enumerate(contagens):
    ax.text(valor + 60, i, str(valor), va="center")

ax.set_xlim(0, contagens.max() * 1.15)
ax.set_title("Quantidade de registros por tipo de imóvel")
ax.set_xlabel("Quantidade de registros")
ax.set_ylabel("Tipo de imóvel")

plt.tight_layout()
plt.show()

In [ ]:
dados = eda.loc[eda["valor_avaliacao"] > 0].copy()
ordem = ["Simples", "Médio", "Superior"]

fig, ax = plt.subplots(figsize=(8, 5))

sns.boxplot(
    data=dados,
    x="padrao_acabamento",
    y="valor_avaliacao",
    order=ordem,
    color="lightsteelblue",
    ax=ax
)

contagens = dados["padrao_acabamento"].value_counts()

ax.set_xticks(range(len(ordem)))
ax.set_xticklabels([
    f"{categoria}\n(n={contagens.get(categoria, 0):,})"
    for categoria in ordem
])

ax.set_yscale("log")
ax.set_title("Valor de avaliação por padrão de acabamento")
ax.set_xlabel("Padrão de acabamento")
ax.set_ylabel("Valor de avaliação (R$; escala logarítmica)")

plt.tight_layout()
plt.show()

In [ ]:
geo = eda[["bairro"]].copy()

geo["sem_coordenadas"] = faltantes[
    ["latitude", "longitude"]
].any(axis=1)

resumo_geo = (
    geo.groupby("bairro")
    .agg(
        n=("sem_coordenadas", "size"),
        faltantes_pct=("sem_coordenadas", lambda s: s.mean() * 100)
    )
)

bairros_top = eda["bairro"].value_counts().head(15).index

dados = (
    resumo_geo.loc[bairros_top]
    .sort_values("faltantes_pct")
)

fig, ax = plt.subplots(figsize=(10, 7))

ax.barh(
    range(len(dados)),
    dados["faltantes_pct"],
    color="steelblue"
)

ax.set_yticks(range(len(dados)))
ax.set_yticklabels([
    f"{bairro} (n={int(n)})"
    for bairro, n in zip(dados.index, dados["n"])
])

for i, valor in enumerate(dados["faltantes_pct"]):
    ax.text(valor + 1, i, f"{valor:.1f}%", va="center")

ax.set_xlim(0, 110)
ax.set_title("Ausência de coordenadas nos 15 bairros mais frequentes")
ax.set_xlabel("Registros sem uma ou ambas as coordenadas (%)")
ax.set_ylabel("Bairro")

plt.tight_layout()
plt.show()